In [ ]:
!pip uninstall -y torch torchvision torchaudio ultralytics -q
!pip install -q \
torch==2.6.0+cu126 \
torchvision==0.21.0+cu126 \
torchaudio==2.6.0+cu126 \
--index-url https://download.pytorch.org/whl/cu126

!pip install -q ultralytics==8.4.78 mlflow opencv-python

In [73]:
import os
import cv2
import yaml
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import mlflow

In [ ]:
ROOT = "/kaggle/input/datasets/cleristonmartinelo/foodsegmentation"

FULL_DATASET = "/kaggle/working/full_yolo_dataset"

EXPERIMENTS = [500, 700, 1500, 2000,2500,3000,4000, 4500, 5000 ]
RUNS_PER_SIZE = 1

MODEL_NAME = "yolo11n-seg.pt"
EPOCHS = 50
BATCH = 8
IMGSZ = 640

In [ ]:
MLFLOW_USER = "mlflowsuperuser"
MLFLOW_PASS = "wfu:1H6B_pFLe',2\""
MLFLOW_HOST = "cleristonm.duckdns.org"

TRACKING_URI = f"http://{MLFLOW_USER}:{MLFLOW_PASS}@{MLFLOW_HOST}:5000"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("foodseg-dataset-scaling_v2")

<Experiment: artifact_location='/mlflow/artifacts/4', creation_time=1783036906315, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1783036906315, lifecycle_stage='active', name='foodseg-dataset-scaling', tags={}, trace_location=None, workspace='default'>

In [76]:
def mask_to_yolo_multiclass(mask_path, output_txt):

    mask = np.array(Image.open(mask_path))
    classes = np.unique(mask)

    with open(output_txt, "w") as f:
        for cls_id in classes:

            if cls_id == 0:
                continue

            binary_mask = (mask == cls_id).astype(np.uint8)

            contours, _ = cv2.findContours(
                binary_mask,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            for contour in contours:

                contour = contour.squeeze()

                if len(contour.shape) < 2:
                    continue

                h, w = mask.shape

                contour = contour.astype(float)
                contour[:, 0] /= w
                contour[:, 1] /= h

                points = contour.flatten().tolist()

                if len(points) >= 6:
                    line = f"{cls_id - 1} " + " ".join(map(str, points))
                    f.write(line + "\n")

In [ ]:
def convert_full_dataset():

    if os.path.exists(FULL_DATASET):
        shutil.rmtree(FULL_DATASET)

    os.makedirs(f"{FULL_DATASET}/images/train", exist_ok=True)
    os.makedirs(f"{FULL_DATASET}/images/val", exist_ok=True)
    os.makedirs(f"{FULL_DATASET}/labels/train", exist_ok=True)
    os.makedirs(f"{FULL_DATASET}/labels/val", exist_ok=True)

    train_images = sorted(Path(f"{ROOT}/Images/img_dir/train").glob("*.jpg"))
    val_images = sorted(Path(f"{ROOT}/Images/img_dir/test").glob("*.jpg"))

    print("Train source:", len(train_images))
    print("Val source:", len(val_images))

    for split_name, image_list in [("train", train_images), ("val", val_images)]:

        source_split = "train" if split_name == "train" else "test"

        for img_path in image_list:

            img_name = img_path.name
            mask_name = Path(img_name).stem + ".png"

            mask_path = f"{ROOT}/Images/ann_dir/{source_split}/{mask_name}"

            shutil.copy(
                img_path,
                f"{FULL_DATASET}/images/{split_name}/{img_name}"
            )

            output_txt = f"{FULL_DATASET}/labels/{split_name}/{Path(img_name).stem}.txt"

            mask_to_yolo_multiclass(mask_path, output_txt)

convert_full_dataset()
from pathlib import Path

train_check = list(Path(f"{FULL_DATASET}/images/train").glob("*"))
val_check = list(Path(f"{FULL_DATASET}/images/val").glob("*"))

print("Converted train:", len(train_check))
print("Converted val:", len(val_check))

print("Example train files:", train_check[:5])
print("Example val files:", val_check[:5])

Train source: 4983
Val source: 2135


In [ ]:
def create_subset(limit_train, seed):

    subset_path = f"/kaggle/working/subset_{limit_train}_{seed}"

    if os.path.exists(subset_path):
        shutil.rmtree(subset_path)

    os.makedirs(f"{subset_path}/images/train", exist_ok=True)
    os.makedirs(f"{subset_path}/images/val", exist_ok=True)
    os.makedirs(f"{subset_path}/labels/train", exist_ok=True)
    os.makedirs(f"{subset_path}/labels/val", exist_ok=True)

    train_imgs = list(Path(f"{FULL_DATASET}/images/train").glob("*.jpg"))
    val_imgs = list(Path(f"{FULL_DATASET}/images/val").glob("*.jpg"))
    
    limit_train = min(limit_train, len(train_imgs))

    print("Train images:", len(train_imgs))
    print("Val images:", len(val_imgs))

    if len(train_imgs) == 0:
        raise Exception("FULL_DATASET train folder is empty")

    random.seed(seed)
    random.shuffle(train_imgs)
    random.shuffle(val_imgs)

    val_limit = max(
        1,
        int(limit_train * (len(val_imgs) / len(train_imgs)))
    )

    selected_train = train_imgs[:limit_train]
    selected_val = val_imgs[:val_limit]

    for img_path in selected_train:

        label_path = Path(
            str(img_path).replace("/images/", "/labels/")
        ).with_suffix(".txt")

        shutil.copy(img_path, f"{subset_path}/images/train/{img_path.name}")
        shutil.copy(label_path, f"{subset_path}/labels/train/{label_path.name}")

    for img_path in selected_val:

        label_path = Path(
            str(img_path).replace("/images/", "/labels/")
        ).with_suffix(".txt")

        shutil.copy(img_path, f"{subset_path}/images/val/{img_path.name}")
        shutil.copy(label_path, f"{subset_path}/labels/val/{label_path.name}")

    yaml_data = {
        "path": subset_path,
        "train": "images/train",
        "val": "images/val",
        "nc": 103
    }

    yaml_path = f"{subset_path}/dataset.yaml"

    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f)

    return yaml_path, limit_train

In [ ]:
history = []

for sample_size in EXPERIMENTS:

    for run_number in range(RUNS_PER_SIZE):

        seed = random.randint(1, 999999)
        
        yaml_path, actual_sample_size = create_subset(sample_size, seed)

        run_name = f"{sample_size}_run_{run_number+1}"
                
        
        with mlflow.start_run(run_name=run_name):

            mlflow.log_params({
                "requested_sample_size": sample_size,
                "actual_sample_size": actual_sample_size,
                "seed": seed,
                "epochs": EPOCHS,
                "batch": BATCH,
                "imgsz": IMGSZ
            })

            model = YOLO(MODEL_NAME)

            model.train(
                data=yaml_path,
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH,
                seed=seed,
                project="/kaggle/working/runs",
                name=run_name
            )

            metrics = model.val()

            mlflow.log_metrics({
                "box_map50": float(metrics.box.map50),
                "seg_map50": float(metrics.seg.map50),
                "precision": float(metrics.box.mp),
                "recall": float(metrics.box.mr)
            })

            best_model = f"/kaggle/working/runs/{run_name}/weights/best.pt"

            if os.path.exists(best_model):
                mlflow.log_artifact(best_model)

            history.append({
                "sample_size": sample_size,
                "run": run_number + 1,
                "seed": seed,
                "box_map50": float(metrics.box.map50),
                "seg_map50": float(metrics.seg.map50),
                "precision": float(metrics.box.mp),
                "recall": float(metrics.box.mr)
            })

In [ ]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "/kaggle/working/experiments_history.csv",
    index=False
)

print(history_df)

with mlflow.start_run(run_name="global_history"):
    mlflow.log_artifact("/kaggle/working/experiments_history.csv")

In [ ]:
history_df = pd.DataFrame(history)

history_df.to_csv("/kaggle/working/experiments_history.csv", index=False)

print(history_df)

mlflow.log_artifact("/kaggle/working/experiments_history.csv")

In [ ]:
print(ROOT)

train_path = Path(f"{ROOT}/Images/img_dir/train")
val_path = Path(f"{ROOT}/Images/img_dir/test")

print("Train exists:", train_path.exists())
print("Val exists:", val_path.exists())

print("Train glob count:", len(list(train_path.glob("*.jpg"))))
print("Val glob count:", len(list(val_path.glob("*.jpg"))))